# FSCT 8561 — Lab 1
## Socket Programming — Stateful Client-Server Chat Application

**Course:** FSCT 8561 — Security Applications  
**Student:** Billy Zandan  
**Lab Duration:** 3 hours

### Overview
This lab builds directly on Lab 0.

In Lab 0, your application used:

> **Connect → Send ONE message → Receive ONE reply → Close**

In Lab 1, you will transform that simple program into a **persistent, stateful client-server chat application**.

You will add:

- multiple messages,
- user input,
- a simple application-layer protocol,
- session state,
- validation,
- error handling,
- graceful disconnect,
- security analysis.


## Learning Objectives

By the end of this lab, you should be able to:

- design a simple application-layer protocol on top of TCP,
- maintain session state across multiple messages,
- keep a TCP connection open,
- use loops and conditional logic,
- validate incoming messages,
- handle malformed input and unexpected disconnects,
- terminate a session cleanly,
- analyze security implications of stateful network applications.


## Prerequisite

You must have completed **Lab 0**.

Start from the working `server.py` and `client.py` you created in Lab 0.


## Required Reading & Tutorials
- 	Mastering Python for Networking and Security – Chapter 3
    -    	https://learning.oreilly.com/library/view/mastering-python-for/9781839217166/
    -   	Source code on: https://github.com/PacktPublishing

-   Mastering-Python-for-Networking-and-Security-Second-Edition
- 	Socket programing tutorials
    - 	Official Python Documentation – socket Module
       -      https://docs.python.org/3/library/socket.html
    - 	Real Python – Socket Programming in Python
       -      https://realpython.com/python-sockets/
    - 	GeeksforGeeks – Socket Programming in Python
       -      https://www.geeksforgeeks.org/socket-programming-python/
    - 	DigitalOcean – How To Use Sockets in Python 3


# Part 1 — New Python Concepts

Lab 1 introduces a few new Python ideas.

### `input()`
Allows a user to type a value:


In [ ]:
message = input("Enter a message: ")
print("You typed:", message)


### `while True`
Repeats code until you explicitly stop it:


In [ ]:
count = 0

while True:
    print("Loop is running")
    count = count + 1

    if count == 3:
        break


### `if / elif / else`
Allows a program to make decisions:


In [ ]:
command = "HELLO"

if command == "HELLO":
    print("Start session")
elif command == "MSG":
    print("Process message")
else:
    print("Unknown command")


### `.split()`
Your protocol will use:

```text
COMMAND|DATA
```

You can separate it with:


In [ ]:
message = "HELLO|Maryam"

command, data = message.split("|", 1)

print("Command:", command)
print("Data:", data)


### `try / except`
Used to handle errors without immediately crashing:


In [ ]:
try:
    number = int("abc")
except ValueError:
    print("Invalid value")


# Part 2 — Start with Your Lab 0 Code

Before adding anything new:

1. Run your Lab 0 server.
2. Run your Lab 0 client.
3. Confirm one message is sent and one response is received.

Do not continue until Lab 0 works.


# Part 3 — Make the Connection Persistent

The key difference in Lab 1 is that the client and server should **not close after one message**.

You will use a loop.

Conceptually:

```text
Connect
   ↓
Receive message
   ↓
Send response
   ↓
Receive another message
   ↓
Send another response
   ↓
...
   ↓
EXIT
   ↓
Close
```


# Part 4 — Application-Level Protocol

All client messages must follow:

```text
COMMAND|DATA
```

Use these commands:

```text
HELLO|username
MSG|text
EXIT|
```

Examples:

```text
HELLO|Maryam
MSG|Hello everyone
MSG|How are you?
EXIT|
```

The server responds with:

```text
OK|
```

or

```text
ERROR|reason
```


# Part 5 — Build the Stateful Server

Create or update:

```text
server.py
```

**Textbook reference:** *Mastering Python for Networking and Security, 2nd Ed.*, Chapter 3.  
The textbook introduces `socket.socket(socket.AF_INET, socket.SOCK_STREAM)` as the standard way to open a TCP socket. It describes `bind()`, `listen()`, `accept()`, `recv()`, and `send()` as the core server-side methods. The `listen(n)` method queues up to n connections. The server below uses these methods directly and adds a session state machine on top.


In [ ]:
"""
Lab 1 — Stateful TCP Server (Single Client)
FSCT 8561 — Security Applications

Protocol:
    Client → Server : COMMAND|DATA
    Server → Client : OK|info  or  ERROR|reason

Commands:
    HELLO|username   — start session, register username
    MSG|text         — send a message (requires prior HELLO)
    EXIT|            — gracefully end session
"""

import socket
import datetime

# Configuration
HOST = "127.0.0.1"
PORT = 12345
MAX_MSG_LEN = 200   # maximum allowed message body length


def log(label, text):
    """Print a timestamped log line to the console."""
    timestamp = datetime.datetime.now().strftime("%H:%M:%S")
    print(f"[{timestamp}] {label}: {text}")


def handle_client(client_socket, client_address):
    """
    Manage one complete client session.
    State: username is None until a valid HELLO is received.
    """
    log("CONNECT", f"New connection from {client_address}")

    username = None      # session state: not authenticated yet
    connected = True

    while connected:
        try:
            # Receive up to 1024 bytes; strip trailing whitespace/newlines
            data = client_socket.recv(1024)

            # Empty bytes means the client closed the connection
            if not data:
                log("DISCONNECT", f"Client {client_address} disconnected unexpectedly")
                break

            # Decode and strip so terminal newlines don't break matching
            message = data.decode("utf-8", errors="replace").strip()
            log("RECV", message)

            # Format validation — must contain pipe character
            if "|" not in message:
                client_socket.send("ERROR|Invalid format — expected COMMAND|DATA".encode())
                log("ERROR", "Invalid format — no pipe character")
                continue

            command, content = message.split("|", 1)
            command = command.upper()   # normalise to uppercase

            # HELLO — register username
            if command == "HELLO":
                if content.strip() == "":
                    client_socket.send("ERROR|Username required".encode())
                    log("ERROR", "HELLO with empty username")
                else:
                    username = content.strip()
                    response = f"OK|Hello {username}"
                    client_socket.send(response.encode())
                    log("SESSION", f"Username registered: {username}")

            # MSG — send a chat message
            elif command == "MSG":
                if username is None:
                    # Data before HELLO — reject
                    client_socket.send("ERROR|Send HELLO first to start a session".encode())
                    log("ERROR", "MSG received before HELLO")
                elif content == "":
                    client_socket.send("ERROR|Message cannot be empty".encode())
                    log("ERROR", f"{client_address} sent empty message")
                elif len(content) > MAX_MSG_LEN:
                    client_socket.send(
                        f"ERROR|Message too long — max {MAX_MSG_LEN} characters".encode()
                    )
                    log("ERROR", f"Oversized message ({len(content)} chars) from {username}")
                else:
                    log("MSG", f"{username}: {content}")
                    client_socket.send(
                        f"OK|Message received from {username}".encode()
                    )

            # EXIT — clean disconnect
            elif command == "EXIT":
                client_socket.send("OK|Goodbye".encode())
                log("EXIT", f"{username or client_address} disconnected cleanly")
                connected = False

            # Unknown command
            else:
                client_socket.send(f"ERROR|Unknown command: {command}".encode())
                log("ERROR", f"Unknown command '{command}' from {username or client_address}")

        except ConnectionResetError:
            log("DISCONNECT", f"Connection reset by {client_address}")
            break
        except OSError as exc:
            log("ERROR", f"Socket error: {exc}")
            break
        except Exception as exc:
            log("ERROR", f"Unexpected error: {exc}")
            break

    client_socket.close()
    log("CLOSED", f"Socket closed for {client_address}")


def main():
    server_socket = socket.socket(socket.AF_INET, socket.SOCK_STREAM)

    # Allow address reuse immediately after restart
    server_socket.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)

    server_socket.bind((HOST, PORT))
    server_socket.listen(1)

    log("START", f"Server listening on {HOST}:{PORT}")

    try:
        # Loop forever — after each client disconnects, wait for the next one
        while True:
            log("WAIT", "Waiting for a new connection...")
            client_socket, client_address = server_socket.accept()
            handle_client(client_socket, client_address)
    except KeyboardInterrupt:
        log("STOP", "Server interrupted by user (Ctrl+C)")
    finally:
        server_socket.close()
        log("STOP", "Server socket closed")


if __name__ == "__main__":
    main()


## What the Server Now Does

Compared with Lab 0, the server now:

- stays connected and loops back to accept new clients after each session,
- processes multiple messages per session,
- stores a username as session state,
- knows whether `HELLO` occurred before processing `MSG`,
- validates commands and message length,
- rejects invalid messages with `ERROR|reason`,
- handles unexpected disconnects (`recv()` returns empty bytes),
- exits cleanly after `EXIT|`.

**Textbook note:** Chapter 3 of *Mastering Python for Networking and Security* describes the same `bind()` → `listen()` → `accept()` → `recv()` / `send()` pattern. The book's `tcp_server.py` example shows a basic threaded loop; our implementation adds a state machine on top using `username = None` as the pre-authentication sentinel.


# Part 6 — Build the Stateful Client

Update:

```text
client.py
```

**Note:** The original file was mistakenly named `client..py` (double dot). The corrected filename is `client.py`.


In [ ]:
"""
Lab 1 — Stateful TCP Client
FSCT 8561 — Security Applications

Protocol:
    HELLO|username  — register with the server
    MSG|text        — send a chat message
    EXIT|           — clean disconnect
"""

import socket

HOST = "127.0.0.1"
PORT = 12345


def main():
    client_socket = socket.socket(socket.AF_INET, socket.SOCK_STREAM)

    try:
        client_socket.connect((HOST, PORT))
        print(f"Connected to server at {HOST}:{PORT}")

        # Step 1: Identify ourselves with HELLO
        username = input("Enter your username: ").strip()

        if not username:
            print("Username cannot be empty. Exiting.")
            return

        client_socket.send(f"HELLO|{username}".encode())

        # Read and display the server's response to HELLO
        response = client_socket.recv(1024).decode("utf-8", errors="replace")
        print(f"Server: {response}")

        if response.startswith("ERROR"):
            print("Server rejected username. Exiting.")
            return

        # Step 2: Message loop
        print("\nType a message and press Enter. Type EXIT to disconnect.\n")

        while True:
            try:
                message = input("You: ").strip()
            except EOFError:
                break

            if message.upper() == "EXIT":
                client_socket.send("EXIT|".encode())
                farewell = client_socket.recv(1024).decode("utf-8", errors="replace")
                print(f"Server: {farewell}")
                break

            # Wrap the text in the MSG command
            client_socket.send(f"MSG|{message}".encode())

            # Receive and display the server's acknowledgement
            response = client_socket.recv(1024).decode("utf-8", errors="replace")
            print(f"Server: {response}")

    except ConnectionRefusedError:
        print(f"ERROR: Could not connect to {HOST}:{PORT} — is the server running?")
    except KeyboardInterrupt:
        print("\nInterrupted. Sending EXIT to server...")
        try:
            client_socket.send("EXIT|".encode())
        except OSError:
            pass
    except OSError as exc:
        print(f"Socket error: {exc}")
    finally:
        client_socket.close()
        print("Disconnected.")


if __name__ == "__main__":
    main()


# Part 7 — Test Normal Operation

Run the server first.

Then run the client.

Example session:

```text
Enter your username: Maryam
Server: OK|Hello Maryam

Enter message or type EXIT to leave: Hello
Server: OK|Message received from Maryam

Enter message or type EXIT to leave: How are you?
Server: OK|Message received from Maryam

Enter message or type EXIT to leave: EXIT
Server: OK|Goodbye
Disconnected
```

After the client exits, the server prints:
```text
[HH:MM:SS] WAIT: Waiting for a new connection...
```
and remains running, ready for the next client.


# Part 8 — Robustness Tests

Demonstrate at least **three** of the following:

1. Empty message
2. Message longer than 200 characters
3. Invalid command format
4. `MSG|...` sent before `HELLO`
5. Unexpected client disconnect
6. Unknown command

For some tests, you may temporarily modify the client code to send an invalid protocol message.

Example:

```python
client_socket.send("BADCOMMAND|test".encode())
```

## Results

### Case 1 — Empty Message Body
```
Client sends : MSG|
Server replies: ERROR|Message cannot be empty
```
The client sends `MSG|` when the user presses Enter without typing. The server checks `if content == "":` before processing. Session remains active.

### Case 2 — Oversized Message (>200 characters)
```
Client sends : MSG|AAAA... (201 characters)
Server replies: ERROR|Message too long — max 200 characters
```
The server checks `len(content) > MAX_MSG_LEN`. The session remains active.

### Case 3 — Invalid Command Format (no pipe)
```
Client sends : BADCOMMAND
Server replies: ERROR|Invalid format — expected COMMAND|DATA
```
The server checks `if "|" not in message:` before splitting, protecting against a `ValueError`.

### Case 4 — MSG Before HELLO
```
Client sends : MSG|Hello
Server replies: ERROR|Send HELLO first to start a session
```
The server checks `if username is None:`. The client can still send `HELLO` afterward.

### Case 5 — Unexpected Client Disconnect
When the client process is killed without sending `EXIT|`:
```
recv() returns b""  →  Server logs: DISCONNECT: Client disconnected unexpectedly
```
The server breaks from the loop and waits for the next client.

### Case 6 — Unknown Command
```
Client sends : PING|hello
Server replies: ERROR|Unknown command: PING
```
The `else` branch sends an error. Session stays alive.


# Part 9 — Why `recv(1024)` Is Not a Message Boundary

TCP is a **byte stream**.

This means:

```python
recv(1024)
```

does **not** guarantee that one complete application message arrives in one call.

A long message may be split across multiple `recv()` calls, or multiple small messages may arrive together.

For this lab, the simple protocol is acceptable for learning purposes, but a production system needs proper **message framing**.

**Why this happens:**
- TCP's Nagle algorithm may batch small writes together.
- The OS may return partial data if the network buffers haven't been fully filled.
- Two fast `send()` calls from the client may arrive in one `recv()` on the server.

**Production solution:** Prefix each message with a fixed-length header containing the message length (e.g., a 4-byte big-endian integer). The receiver first reads exactly 4 bytes to get the length, then reads exactly that many bytes for the body.


# Part 10 — Security Reflection

**Q1. What happens if the server crashes while the client is connected?**

When the server process terminates unexpectedly, the OS closes all its file descriptors and sends a TCP FIN (or RST) to the client. On the client side, the next `recv()` call returns `b""` (empty bytes), or raises `ConnectionResetError`. In our implementation, the client's `recv()` in the message loop will either get an empty response or throw an exception caught by `except OSError`. The client prints an error and exits cleanly. Without error handling, the client would hang or crash.

**Q2. How could the server be extended to handle multiple clients?**

The single-client server's `handle_client()` loop occupies the main thread entirely. To support concurrent clients, spawn a `threading.Thread` for each new connection after `accept()`. The main thread immediately returns to `accept()` while each client thread independently runs `handle_client()`. Alternatives include `asyncio` (cooperative multitasking), `multiprocessing`, or I/O multiplexing with `select()`/`poll()`. See the Challenge section for the full implementation.

**Q3. Why can `recv(1024)` split messages unexpectedly?**

TCP is a byte stream protocol, not a message-oriented one. It guarantees delivery and ordering of bytes but not delivery in application-level chunks. Two scenarios arise: (1) **fragmentation** — a message larger than the MTU arrives in multiple `recv()` calls; (2) **coalescing** — two small messages sent quickly arrive in one `recv()` call due to Nagle's algorithm or kernel buffering. The `|` delimiter in our protocol only helps parse a correctly-arrived message; it does not fix partial delivery. A production protocol uses length-prefixed framing.

**Q4. Why does TCP reliability not mean the application is secure?**

TCP reliability means bytes are delivered in order without loss. It does **not** mean confidentiality, integrity against tampering, or authentication of the sender. An attacker with network access can: eavesdrop (read all bytes in plaintext), tamper (modify bytes in transit if the TCP checksum is forged), replay (retransmit a captured session), or impersonate (conduct TCP session hijacking with guessed sequence numbers). Security requires TLS layered on top of TCP, which provides encryption, HMAC integrity, and certificate-based authentication.

**Q5. What new attack surfaces are created by adding usernames and protocol commands?**

- **Username spoofing** — Any client claims any name; no cryptographic proof.
- **Protocol injection** — Crafted inputs (e.g., long usernames, embedded `|`) probe for parsing bugs.
- **State confusion** — Sending `MSG` before `HELLO` tests whether the state machine is enforced.
- **Resource exhaustion** — Many connections without `HELLO` tie up server threads and file descriptors.
- **Information leakage** — Verbose error messages may reveal server internals.

**Q6. How could a malicious client abuse the protocol?**

1. **Username enumeration** — Try `HELLO|admin`, `HELLO|root` to discover active names.
2. **DoS via flooding** — Send messages just under the 200-char limit as fast as possible.
3. **Connection exhaustion** — Open hundreds of TCP connections without sending `HELLO`.
4. **Pipe injection** — Send `HELLO|alice|extra` to test `split()` handling.
5. **Broadcast amplification** (multi-client) — One spammer multiplies load on all connected clients.
6. **Impersonation** — Re-register a username immediately after the legitimate user logs off.

**Q7. How would you add authentication?**

On connect, the server sends a random nonce. The client responds with `HMAC-SHA256(shared_secret, nonce)` inside the `HELLO` message. The server verifies. Without the secret, an attacker cannot forge a valid response. For multi-user systems, store bcrypt-hashed passwords per username; the client sends the password, the server compares. Wrap everything in TLS to prevent the password from being sniffed.

**Q8. How would you add encryption?**

Use Python's built-in `ssl` module to wrap the socket:
```python
import ssl
context = ssl.SSLContext(ssl.PROTOCOL_TLS_SERVER)
context.load_cert_chain("server.crt", "server.key")
secure_socket = context.wrap_socket(server_socket, server_side=True)
```
All `send()` and `recv()` calls are then transparently encrypted. The application-layer protocol (`COMMAND|DATA`) does not change.

**Q9. What validation is performed in your implementation?**

| Check | Location |
|---|---|
| Pipe character present | Server, before `split()` |
| Command is a known value | Server, `if/elif/else` dispatch |
| Username not empty | Server `HELLO` handler; Client before connecting |
| `HELLO` sent before `MSG` | Server, checks `username is None` |
| Message body not empty | Server, `MSG` handler |
| Message body ≤ 200 chars | Server, `MSG` handler |
| Unexpected disconnect | Server, checks `if not data:` |

**Q10. What additional mitigations would be needed for a real deployment?**

1. TLS encryption for all traffic
2. Password-based or token-based authentication
3. Rate limiting (connections per IP, messages per second)
4. Input sanitization (strip control characters, null bytes)
5. Length-prefixed message framing to eliminate stream ambiguity
6. Persistent logging to file with timestamps
7. Per-connection idle timeout (`socket.settimeout()`)
8. Maximum concurrent connection limit
9. Process isolation with minimal OS privileges
10. Regular security audits and dependency updates


# Part 11 — Raw TCP vs. Socket.IO

The textbook (*Mastering Python for Networking and Security*, Chapter 2) shows a Socket.IO implementation using `python-socketio` and `aiohttp`. The book demonstrates a server with `@socket_io.on('message')` decorators and a client that calls `sio.emit('message', {'data': 'my_data'})`. This is a very different model from raw TCP sockets.

Raw TCP sockets use operations such as:

```text
bind()
listen()
accept()
connect()
send()
recv()
```

Socket.IO hides many of these details and provides higher-level event-based communication.

Example concept (from textbook Chapter 2):

```python
sio.emit("message", {"data": "hello"})
```

### Task
Implement stateful chat server using Socket.IO and test it.

See: `socketio_demo/server_socketio.py` and `socketio_demo/client_socketio.py`

### Discussion — Which low-level operations does Socket.IO hide?

| Operation | Raw TCP | Socket.IO |
|---|---|---|
| `bind()` | Explicit | Hidden inside WSGI/aiohttp server |
| `listen()` | Explicit | Hidden |
| `accept()` | Explicit loop | Dispatched by `@sio.event connect` |
| `recv()` | Manual call returning raw bytes | Not visible; event handler called automatically |
| `send()` | `socket.send(bytes)` | `sio.emit("event", dict)` |
| Data encoding | `".encode()"` / `".decode()"` manually | JSON serialization automatic |
| Message framing | Developer's `\|` delimiter | Guaranteed: one `emit` = one received event |
| Connection tracking | Manual `clients` dict | Automatic `sid` per client |
| Multi-client | Requires `threading` explicitly | Built-in (eventlet/asyncio) |
| Reconnection | Not implemented | Built-in reconnection logic |

**Conclusion:** Socket.IO trades low-level visibility for developer convenience. Raw TCP is better for learning and security analysis because every byte is explicit. Socket.IO is better for production apps because it handles framing, routing, and reconnection automatically.

This comparison is for understanding abstraction. The required Lab 1 implementation uses **raw TCP sockets**.


In [ ]:
"""
Lab 1 — Socket.IO Server Demo
FSCT 8561 — Security Applications

Based on the pattern shown in:
  Mastering Python for Networking and Security, 2nd Ed., Chapter 2
  (web_socket_server.py example using socketio + aiohttp)

Install dependencies first:
    pip install python-socketio eventlet
"""

import socketio
import eventlet
import eventlet.wsgi

sio = socketio.Server(cors_allowed_origins="*")
app = socketio.WSGIApp(sio)

# Session state: maps socket ID → username
sessions = {}


@sio.event
def connect(sid, environ):
    """Called automatically when a new client connects."""
    sessions[sid] = None
    print(f"[CONNECT] Client connected: {sid}")


@sio.event
def disconnect(sid):
    """Called automatically when a client disconnects."""
    username = sessions.pop(sid, None)
    print(f"[DISCONNECT] {username or sid} disconnected")


@sio.event
def hello(sid, data):
    """Equivalent to HELLO|username in the raw TCP protocol."""
    username = data.get("username", "").strip()
    if not username:
        sio.emit("error", {"reason": "Username required"}, to=sid)
        return
    sessions[sid] = username
    print(f"[SESSION] Username registered: {username}")
    sio.emit("ok", {"info": f"Hello {username}"}, to=sid)


@sio.event
def message(sid, data):
    """Equivalent to MSG|text in the raw TCP protocol."""
    username = sessions.get(sid)
    if username is None:
        sio.emit("error", {"reason": "Send HELLO first"}, to=sid)
        return
    text = data.get("text", "")
    if not text:
        sio.emit("error", {"reason": "Message cannot be empty"}, to=sid)
        return
    if len(text) > 200:
        sio.emit("error", {"reason": "Message too long"}, to=sid)
        return
    print(f"[MSG] {username}: {text}")
    sio.emit("ok", {"info": f"Message received from {username}"}, to=sid)


@sio.event
def exit(sid, data):
    """Equivalent to EXIT| in the raw TCP protocol."""
    username = sessions.get(sid)
    print(f"[EXIT] {username or sid} is leaving")
    sio.emit("ok", {"info": "Goodbye"}, to=sid)
    sio.disconnect(sid)


# NOTE: Run this script directly from a terminal, not inside the notebook
# python socketio_demo/server_socketio.py
print("Socket.IO server ready — run from terminal: python socketio_demo/server_socketio.py")


# Challenge — Support Multiple Clients

Your current Lab 1 server supports only **one client at a time**. In a real chat application, multiple users should be able to connect to the same server.

For this challenge, extend your Lab 1 application so that the server can support **multiple clients concurrently**.

## Challenge Requirements

Modify your application so that:

1. At least **two clients** can connect to the server at the same time.
2. Each client provides a username using:

```text
HELLO|username
```

3. The server maintains the username associated with each connected client.
4. A client can send:

```text
MSG|Hello everyone!
```

5. The server forwards the message to the other connected client(s), including the sender's username:

```text
Maryam: Hello everyone!
```

6. `EXIT|` disconnects only that client while the server continues running for other users.
7. The server handles an unexpected client disconnect without crashing.

## Hint — Concurrency with Threads

Your current server becomes occupied while communicating with one client.

Research how Python's `threading` module can allow the server to handle each connected client independently.

You may begin by exploring:

```python
import threading
```

and:

```python
threading.Thread(...)
```

Do not simply copy a complete multi-client chat program. Your goal is to understand how concurrency changes the design of the Lab 1 server.

## Demonstration

Run the programs in three terminals:

```text
Terminal 1 → challenge/server.py

Terminal 2 → challenge/client.py → Alice

Terminal 3 → challenge/client.py → Bob
```

Demonstrate that Alice and Bob can exchange messages through the server.


In [ ]:
"""
Lab 1 Challenge — Multi-Client Threaded TCP Server
FSCT 8561 — Security Applications

Each connected client gets its own thread.
MSG commands are broadcast to every OTHER connected client.

Textbook reference: Mastering Python for Networking and Security, Ch. 2
  — threading module for managing concurrent client connections.

Run from terminal: python challenge/server.py
"""

import socket
import threading
import datetime

HOST = "127.0.0.1"
PORT = 12346          # different port so it can run alongside Lab 1 server
MAX_MSG_LEN = 200

# Shared state — protected by a lock to avoid race conditions
clients_lock = threading.Lock()
clients = {}   # {socket: username | None}


def log(label, text):
    timestamp = datetime.datetime.now().strftime("%H:%M:%S")
    print(f"[{timestamp}] [{label}] {text}")


def broadcast(sender_socket, message):
    """Send a message to every connected client except the sender."""
    dead_sockets = []
    with clients_lock:
        for sock in list(clients.keys()):
            if sock is sender_socket:
                continue
            try:
                sock.send(message.encode())
            except OSError:
                dead_sockets.append(sock)
    for sock in dead_sockets:
        remove_client(sock)


def remove_client(sock):
    """Safely remove a client from the shared registry and close its socket."""
    with clients_lock:
        username = clients.pop(sock, None)
    try:
        sock.close()
    except OSError:
        pass
    if username:
        log("LEFT", f"{username} has left the server")
        broadcast(sock, f"SERVER|{username} has left the chat")


def handle_client(client_socket, client_address):
    """Handle one client session in its own thread."""
    log("CONNECT", f"New connection from {client_address}")
    with clients_lock:
        clients[client_socket] = None

    connected = True
    while connected:
        try:
            data = client_socket.recv(1024)
            if not data:
                log("DISCONNECT", f"{client_address} disconnected unexpectedly")
                break

            message = data.decode("utf-8", errors="replace").strip()
            log("RECV", f"{client_address} → {message}")

            if "|" not in message:
                client_socket.send("ERROR|Invalid format — expected COMMAND|DATA".encode())
                continue

            command, content = message.split("|", 1)
            command = command.upper()

            with clients_lock:
                username = clients.get(client_socket)

            if command == "HELLO":
                if content.strip() == "":
                    client_socket.send("ERROR|Username required".encode())
                else:
                    new_name = content.strip()
                    with clients_lock:
                        existing_names = list(clients.values())
                    if new_name in existing_names:
                        client_socket.send(
                            f"ERROR|Username '{new_name}' is already taken".encode()
                        )
                    else:
                        with clients_lock:
                            clients[client_socket] = new_name
                        client_socket.send(f"OK|Hello {new_name}".encode())
                        log("SESSION", f"User registered: {new_name}")
                        broadcast(client_socket, f"SERVER|{new_name} has joined the chat")

            elif command == "MSG":
                if username is None:
                    client_socket.send("ERROR|Send HELLO first".encode())
                elif content == "":
                    client_socket.send("ERROR|Message cannot be empty".encode())
                elif len(content) > MAX_MSG_LEN:
                    client_socket.send(
                        f"ERROR|Message too long — max {MAX_MSG_LEN} characters".encode()
                    )
                else:
                    client_socket.send("OK|Message sent".encode())
                    log("MSG", f"{username}: {content}")
                    broadcast(client_socket, f"MSG|{username}: {content}")

            elif command == "EXIT":
                client_socket.send("OK|Goodbye".encode())
                log("EXIT", f"{username or client_address} exited cleanly")
                connected = False

            else:
                client_socket.send(f"ERROR|Unknown command: {command}".encode())

        except ConnectionResetError:
            log("DISCONNECT", f"Connection reset by {client_address}")
            break
        except OSError as exc:
            log("ERROR", f"Socket error for {client_address}: {exc}")
            break
        except Exception as exc:
            log("ERROR", f"Unexpected: {exc}")
            break

    remove_client(client_socket)


def main():
    server_socket = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    server_socket.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
    server_socket.bind((HOST, PORT))
    server_socket.listen(5)
    log("START", f"Multi-client server listening on {HOST}:{PORT}")

    try:
        while True:
            client_socket, client_address = server_socket.accept()
            # Spawn a dedicated thread for each new client
            thread = threading.Thread(
                target=handle_client,
                args=(client_socket, client_address),
                daemon=True
            )
            thread.start()
            log("THREAD", f"Started thread for {client_address} (active: {threading.active_count() - 1})")
    except KeyboardInterrupt:
        log("STOP", "Server interrupted by user")
    finally:
        server_socket.close()
        log("STOP", "Server socket closed")


# NOTE: Run from terminal — python challenge/server.py
print("Challenge server ready — run from terminal: python challenge/server.py")


## Challenge Reflection

**Q1. Why could the original Lab 1 server handle only one client at a time?**

The original server called `accept()` once, then entered `handle_client()` which ran a `while connected:` loop. The server's main thread was entirely occupied inside that loop and could not call `accept()` again until the current client disconnected. Any second client would be held in the OS backlog queue or refused.

**Q2. What problem does threading solve?**

Threading allows the server to run each client's `handle_client()` loop in its own independent OS thread. After `accept()`, the main thread immediately hands the socket to a new `Thread` and loops back to `accept()`. Multiple clients are served in parallel, each blocked only on their own `recv()`, not on each other.

**Q3. What happens to the other clients when one client disconnects?**

Nothing — only that client's thread exits and that socket closes. The server's main thread keeps looping on `accept()`. All other client threads continue running their own `while connected:` loops normally. The server broadcasts a `SERVER|username has left the chat` message to notify the remaining clients.

**Q4. What additional security risks arise when multiple clients share the same server?**

- **Cross-client information leakage** — All broadcast messages are visible to every connected client.
- **Impersonation** — Any client can register as any username not currently active.
- **Broadcast amplification** — One spammer multiplies load onto all clients.
- **Race conditions** — Multiple threads sharing `clients` require careful locking (`threading.Lock`).
- **Thread exhaustion DoS** — An attacker opening hundreds of connections without `HELLO` exhausts server threads and memory.

**Q5. What could happen if two clients try to use the same username?**

Without the duplicate check, the second client silently overwrites the first in the `clients` dict. This causes identity confusion during broadcasts and broken cleanup on disconnect. Our server prevents this: `if new_name in existing_names:` → `ERROR|Username 'X' is already taken`.

**Q6. How would you prevent one user from impersonating another user?**

1. **Password + bcrypt** — Require `HELLO|username|password`; server stores hashed passwords.
2. **Session tokens** — Server issues a random token on login; future connections must present it.
3. **HMAC challenge-response** — Server sends a nonce; client responds with `HMAC-SHA256(secret, nonce)`.
4. **TLS client certificates** — Certificate signed by a trusted authority cryptographically proves identity before any application data is sent.


## Security Analysis (300–400 words)

### Attack Surfaces Introduced by Protocol and State

The Lab 1 implementation introduces several security concerns that do not exist in a simple one-shot echo server.

**Protocol Injection and Parsing Vulnerabilities**

The `COMMAND|DATA` format is parsed using Python's `.split("|", 1)`. The `maxsplit=1` parameter prevents the data field from splitting on embedded pipes, but a crafted input such as `HELLO|alice\x00bob` (containing a null byte) could corrupt username state in languages that use null-terminated strings. Python's string handling is null-safe, but the potential for protocol injection increases whenever the server interprets user-supplied data as control information. The server normalizes commands to uppercase (`command.upper()`), preventing case-variation attacks, but does not sanitize whitespace or control characters in the username or message body.

**State Machine Abuse**

The server enforces a minimal state machine: `MSG` before `HELLO` is rejected. However, the server allows `HELLO` to be sent multiple times, each time overwriting the stored username. A connected client can therefore change its apparent identity mid-session. In the multi-client challenge, duplicate usernames are prevented, but a client can still abandon its registered name and re-register with a new one without authenticating.

**Denial of Service via Connection Exhaustion**

The single-client server accepts only one connection; the multi-client server spawns one OS thread per client. An attacker can open many simultaneous TCP connections without sending `HELLO`, occupying threads, sockets, and memory. Python threads carry significant per-thread overhead (~8 MB stack on Linux). At a few hundred connections, the server could exhaust available memory or hit the OS thread limit, causing it to stop accepting new legitimate clients.

**Cleartext Transmission**

All data — usernames and message content — is transmitted as plaintext UTF-8 over TCP. Any attacker with access to the network path (via ARP spoofing on a LAN or a compromised router) can read all messages with a packet sniffer such as Wireshark. TCP's reliability guarantees do not address confidentiality. In a real deployment, TLS via Python's `ssl` module must wrap the socket before any application data is sent.

**Limitations of TCP-Level Security**

TCP provides checksums for detecting accidental corruption but no protection against deliberate manipulation by an adversary with network access. Without TLS, there is no mechanism for the client to verify that it has connected to a legitimate server rather than a man-in-the-middle attacker.

**Appropriate Mitigations**

The most impactful mitigations are: (1) wrapping sockets with `ssl.wrap_socket()` to encrypt traffic; (2) adding a shared-secret HMAC challenge in the `HELLO` handshake to authenticate clients; (3) implementing a per-IP connection limit and rate limiter to prevent DoS; (4) adding a server-side timeout (`socket.settimeout()`) to disconnect idle clients and free resources. These changes address the highest-priority risks without fundamentally redesigning the protocol.


# Deliverables

Submit:

- `server.py` ✅
- `client.py` ✅ (note: original was named `client..py` — corrected)
- a recording demonstrating:
  - connection,
  - multiple messages,
  - protocol behavior,
  - at least three robustness tests,
  - clean disconnection,
- a 300–400 word security analysis ✅ (see Part 10 / Security Analysis above)
- reflection answers ✅ (see Part 10)

Challenge also includes:

- `challenge/server.py` ✅
- `challenge/client.py` ✅
- screenshots or a short recording showing at least two clients connected at the same time
- answers to the Challenge Reflection questions ✅ (see Challenge Reflection above)

Submit one PDF containing the security analysis, reflection answers, and links to technical artifacts.

### Filename

```text
Lab1-Billy-Zandan-[StudentNumber].pdf
```

### File Structure

```text
Lab1/
├── Lab1.ipynb                      ← this file
├── server.py                       ← stateful single-client server
├── client.py                       ← stateful client (fixed filename)
├── client..py                      ← original (preserved)
├── challenge/
│   ├── server.py                   ← threaded multi-client server
│   └── client.py                   ← client with receive thread
└── socketio_demo/
    ├── server_socketio.py          ← Socket.IO server demo
    └── client_socketio.py          ← Socket.IO client demo
```
